In [25]:
# ==============================================================================
# Cell 1 — Install Packages
# ==============================================================================
# transformers: wav2vec2
# No pyloudnorm, no silero, no torch.hub needed
!pip install -q transformers librosa pydub nest-asyncio scikit-learn seaborn tqdm

import nest_asyncio
nest_asyncio.apply()
print("✓ Packages ready.")


✓ Packages ready.


In [26]:
# ==============================================================================
# Cell 2 — Global Configuration
# ==============================================================================
import os, random, math, shutil
import numpy as np
import pandas as pd
from collections import Counter
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchaudio
import torchaudio.functional as AF
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark     = True
USE_COMPILE = False    # set False if torch < 2.0 or on Windows

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPUS = torch.cuda.device_count()
print(f"Device : {DEVICE}  |  GPUs: {N_GPUS}")
if DEVICE.type == "cuda":
    for i in range(N_GPUS):
        print(f"  GPU {i} : {torch.cuda.get_device_name(i)}")

# ── Audio ─────────────────────────────────────────────────────────────────────
SR             = 16_000
CLIP_LEN       = 5.0
TARGET_SAMPLES = int(SR * CLIP_LEN)

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE       = 64
EPOCHS           = 100
PATIENCE         = 20
LR               = 3e-4
WEIGHT_DECAY     = 1e-4
MULTICROP_STRIDE = int(2.0 * SR)
WARMUP_EPOCHS    = 5

# ── ArcFace ───────────────────────────────────────────────────────────────────
AAM_SCALE  = 20.0
AAM_MARGIN = 0.35

# ── Loss weights ─────────────────────────────────────────────────────────────
LAMBDA_ARCFACE = 1.0    # ArcFace margin loss (main)
LAMBDA_FOCAL   = 0.3    # Focal loss auxiliary — re-weights hard/minority samples
LAMBDA_SCL     = 0.015  # Supervised contrastive loss — tightens intra-class clusters
FOCAL_GAMMA    = 2.0    # Focal focusing parameter (2.0 = standard, per ICDR paper)
SCL_TEMP       = 0.07   # Contrastive temperature (standard value)
MIXUP_PROB     = 0.3    # Probability of applying MixUp to a batch
MIXUP_ALPHA    = 0.4    # Beta(alpha, alpha) — keeps mixes near one class endpoint

# ── Augmentation ─────────────────────────────────────────────────────────────
REAL_NOISE_SNR_MIN = 15.0
REAL_NOISE_SNR_MAX = 30.0

AUG_PROB = {
    "scared":        0.60,
    "needs":         0.40,
    "physical_pain": 0.70,
    "burping":       0.85,
}

# ── Kaggle paths ──────────────────────────────────────────────────────────────
LOCAL_DATA_ROOT = "/kaggle/input/datasets/karimeletriby/infant-cry-classification-4-classes/audio"
LOCAL_CKPT_DIR  = "/kaggle/working/checkpoints"
MODEL_PATH      = os.path.join(LOCAL_CKPT_DIR, "best_w2v_ecapa.pt")
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)

# ── Classes (4-class clean dataset) ──────────────────────────────────────────
CLASSES   = ["scared", "needs", "physical_pain", "burping"]
CLASS_MAP = {c: c for c in CLASSES}
CLASS2IDX = {c: i for i, c in enumerate(CLASSES)}
IDX2CLASS = {i: c for c, i in CLASS2IDX.items()}
N_CLASSES = len(CLASSES)

AUG_PROB_BY_IDX = {CLASS2IDX[c]: p for c, p in AUG_PROB.items()}

print("\n✓ Configuration complete.")
print(f"  Classes ({N_CLASSES}): {CLASSES}")
print(f"  Losses : ArcFace x{LAMBDA_ARCFACE} | Focal x{LAMBDA_FOCAL} (γ={FOCAL_GAMMA}) | SCL x{LAMBDA_SCL} (T={SCL_TEMP})")
print(f"  MixUp  : prob={MIXUP_PROB}  alpha={MIXUP_ALPHA}")

Device : cuda  |  GPUs: 2
  GPU 0 : Tesla T4
  GPU 1 : Tesla T4

✓ Configuration complete.
  Classes (4): ['scared', 'needs', 'physical_pain', 'burping']
  Losses : ArcFace x1.0 | Focal x0.3 (γ=2.0) | SCL x0.015 (T=0.07)
  MixUp  : prob=0.3  alpha=0.4


In [27]:
# ==============================================================================
# Cell 3 — Dataset Verification
# ==============================================================================
print(f"Verifying: {LOCAL_DATA_ROOT}")
total_files = 0
for cls in CLASSES:
    cls_path = os.path.join(LOCAL_DATA_ROOT, cls)
    if not os.path.isdir(cls_path):
        print(f"  ⚠  Missing: {cls_path}"); continue
    n = len([f for f in os.listdir(cls_path)
             if f.lower().endswith((".wav", ".mp3", ".ogg", ".m4a", ".flac"))])
    total_files += n
    print(f"  {cls:15s} : {'█'*(n//50)} {n}")
print(f"\n  Total audio files: {total_files}")


Verifying: /kaggle/input/datasets/karimeletriby/infant-cry-classification-4-classes/audio
  scared          : █████████████ 684
  needs           : ██████████████████████████████████████████████████████████████ 3103
  physical_pain   : ███████████████ 774
  burping         : ██████ 310

  Total audio files: 4871


In [28]:
# ==============================================================================
# Cell 4 — Dataset Fingerprinting & Provenance Analysis
# ==============================================================================
import hashlib
import soundfile as sf
from collections import Counter

def file_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

# ── Collect all files ─────────────────────────────────────────────────────────
print("Collecting file paths...")
raw_files, raw_labels = [], []
for cls in CLASSES:
    cls_dir = os.path.join(LOCAL_DATA_ROOT, cls)
    if not os.path.isdir(cls_dir): continue
    for f in os.listdir(cls_dir):
        if f.lower().endswith((".wav", ".mp3", ".ogg", ".m4a", ".flac")):
            raw_files.append(os.path.join(cls_dir, f))
            raw_labels.append(cls)

print(f"Total files found : {len(raw_files)}")

# ── 1. Duplicate detection ────────────────────────────────────────────────────
print("\n── Duplicate Detection ──────────────────────────────────────")
hashes     = {}
duplicates = 0
for path in tqdm(raw_files, desc="Hashing", ncols=70):
    try:
        h = file_hash(path)
        if h in hashes: duplicates += 1
        else:           hashes[h] = path
    except Exception as e:
        print(f"  ⚠ {os.path.basename(path)}: {e}")

print(f"  Unique files     : {len(hashes)}")
print(f"  Duplicates found : {duplicates}")

# ── 2. Duration & sample rate analysis ───────────────────────────────────────
print("\n── Duration Analysis ────────────────────────────────────────")
durations, sample_rates, bad_files = [], [], []
for path in tqdm(raw_files, desc="Inspecting", ncols=70):
    try:
        info = sf.info(path)
        durations.append(info.duration)
        sample_rates.append(info.samplerate)
    except Exception as e:
        bad_files.append((path, str(e)))

durations = np.array(durations)
print(f"  Successfully read : {len(durations)} / {len(raw_files)}")
print(f"  Bad files         : {len(bad_files)}")
print(f"  Duration  mean    : {durations.mean():.2f}s")
print(f"  Duration  std     : {durations.std():.4f}s")
print(f"  Duration  min     : {durations.min():.2f}s")
print(f"  Duration  max     : {durations.max():.2f}s")
print(f"  Sample rates      : {Counter(sample_rates)}")

# ── 3. Per-class duration stats ───────────────────────────────────────────────
print(f"\n── Per-Class Duration Stats ─────────────────────────────────")
print(f"  {'Class':15s} {'n':>6s}  {'mean':>7s}  {'std':>7s}  {'min':>7s}  {'max':>7s}")
print(f"  {'-'*55}")
class_durations = {cls: [] for cls in CLASSES}
for path, label, dur in zip(raw_files, raw_labels, durations):
    class_durations[label].append(dur)
for cls in CLASSES:
    d = np.array(class_durations[cls])
    if len(d) == 0: continue
    print(f"  {cls:15s} {len(d):>6d}  {d.mean():>7.2f}s  {d.std():>7.4f}s  "
          f"{d.min():>7.2f}s  {d.max():>7.2f}s")

# ── 4. Class distribution ─────────────────────────────────────────────────────
print(f"\n── Class Distribution ───────────────────────────────────────")
dist = Counter(raw_labels)
total = sum(dist.values())
for cls in CLASSES:
    n   = dist.get(cls, 0)
    pct = 100 * n / total
    bar = "█" * (n // 50)
    print(f"  {cls:15s} {n:>5d}  ({pct:5.1f}%)  {bar}")

print("\n✓ Fingerprinting complete.")


Total files found : 4871

── Duplicate Detection ──────────────────────────────────────


Hashing: 100%|██████████████████| 4871/4871 [00:04<00:00, 1162.01it/s]


  Unique files     : 4871
  Duplicates found : 0

── Duration Analysis ────────────────────────────────────────


Inspecting: 100%|███████████████| 4871/4871 [00:03<00:00, 1530.14it/s]

  Successfully read : 4871 / 4871
  Bad files         : 0
  Duration  mean    : 4.08s
  Duration  std     : 1.1906s
  Duration  min     : 1.02s
  Duration  max     : 7.05s
  Sample rates      : Counter({16000: 4871})

── Per-Class Duration Stats ─────────────────────────────────
  Class                n     mean      std      min      max
  -------------------------------------------------------
  scared             684     3.79s   0.5912s     1.02s     3.99s
  needs             3103     4.23s   1.2431s     1.08s     7.05s
  physical_pain      774     3.71s   1.1538s     1.08s     7.02s
  burping            310     4.09s   1.4021s     1.20s     6.99s

── Class Distribution ───────────────────────────────────────
  scared            684  ( 14.0%)  █████████████
  needs            3103  ( 63.7%)  ██████████████████████████████████████████████████████████████
  physical_pain     774  ( 15.9%)  ███████████████
  burping           310  (  6.4%)  ██████

✓ Fingerprinting complete.


In [29]:
# ==============================================================================
# Cell 4 — Load File Paths & Stratified Split
# ==============================================================================
all_files, all_labels = [], []
for cls in CLASSES:
    cls_dir = os.path.join(LOCAL_DATA_ROOT, cls)
    if not os.path.isdir(cls_dir): continue
    files = [os.path.join(cls_dir, f) for f in os.listdir(cls_dir)
             if f.lower().endswith((".wav", ".mp3", ".ogg", ".m4a", ".flac"))]
    all_files.extend(files)
    all_labels.extend([CLASS2IDX[cls]] * len(files))

X_train, X_tmp, y_train, y_tmp = train_test_split(
    all_files, all_labels, test_size=0.30, stratify=all_labels, random_state=SEED)
X_val, X_test, y_val, y_test   = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

print(f"Split → Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
for cls in CLASSES:
    idx = CLASS2IDX[cls]
    n_tr = y_train.count(idx); n_v = y_val.count(idx); n_te = y_test.count(idx)
    print(f"  {cls:15s}  train={n_tr:4d}  val={n_v:4d}  test={n_te:4d}")


Split → Train: 3409 | Val: 731 | Test: 731
  scared           train= 479  val= 102  test= 103
  needs            train=2171  val= 466  test= 466
  physical_pain    train= 542  val= 116  test= 116
  burping          train= 217  val=  47  test=  46


In [30]:
# ==============================================================================
# Cell 5 - Audio Helpers, Augmentor & DataLoaders
# ==============================================================================
def load_segment(path, start_sample=None):
    wav, sr = torchaudio.load(path)
    if wav.shape[0] > 1: wav = wav.mean(dim=0, keepdim=True)
    if sr != SR: wav = AF.resample(wav, sr, SR)
    y = wav.squeeze().numpy()
    if start_sample is not None:
        y = y[start_sample : start_sample + TARGET_SAMPLES]
        if len(y) < TARGET_SAMPLES: y = np.pad(y, (0, TARGET_SAMPLES - len(y)))
    elif len(y) > TARGET_SAMPLES:
        start = random.randint(0, len(y) - TARGET_SAMPLES)
        y = y[start : start + TARGET_SAMPLES]
    else:
        y = np.pad(y, (0, TARGET_SAMPLES - len(y)))
    return y

def load_center_crop(path):
    wav, sr = torchaudio.load(path)
    if wav.shape[0] > 1: wav = wav.mean(dim=0, keepdim=True)
    if sr != SR: wav = AF.resample(wav, sr, SR)
    y = wav.squeeze().numpy()
    if len(y) > TARGET_SAMPLES:
        start = (len(y) - TARGET_SAMPLES) // 2
        y = y[start : start + TARGET_SAMPLES]
    elif len(y) < TARGET_SAMPLES:
        y = np.pad(y, (0, TARGET_SAMPLES - len(y)))
    return y

def peak_norm(y):
    peak = np.abs(y).max()
    return y / (peak + 1e-8)

def add_reverb(y, sr=SR):
    rt60  = random.uniform(0.05, 0.3)
    n     = int(rt60 * sr)
    rir   = np.exp(-3.0 * np.arange(n) / (rt60 * sr)).astype(np.float32)
    rir  /= rir.sum()
    out   = np.convolve(y, rir, mode="full")[:len(y)]
    peak  = np.abs(out).max()
    return (out / (peak + 1e-8)).astype(np.float32)

def build_multicrop_list(file_list, label_list, stride=MULTICROP_STRIDE):
    out_entries, out_labels = [], []
    for path, label in tqdm(zip(file_list, label_list),
                            total=len(file_list), desc="Building index"):
        try:
            info      = torchaudio.info(path)
            n_samples = int(info.num_frames / info.sample_rate * SR)
        except Exception:
            n_samples = TARGET_SAMPLES
        if n_samples <= TARGET_SAMPLES:
            out_entries.append((path, None)); out_labels.append(label)
        else:
            for s in range(0, n_samples - TARGET_SAMPLES + 1, stride):
                out_entries.append((path, s)); out_labels.append(label)
    return out_entries, out_labels

print("Building multi-crop index...")
train_entries, train_entry_labels = build_multicrop_list(X_train, y_train)


class GPUAudioAugmentor(nn.Module):
    def __init__(self, sr=SR, target_samples=TARGET_SAMPLES):
        super().__init__()
        self.sr = sr; self.target_samples = target_samples

    def _ensure_len(self, y):
        if y.size(-1) > self.target_samples: return y[..., :self.target_samples]
        if y.size(-1) < self.target_samples: return F.pad(y, (0, self.target_samples - y.size(-1)))
        return y

    def forward(self, waveforms, labels, aug_probs_by_idx):
        B     = waveforms.size(0)
        probs = torch.tensor([aug_probs_by_idx.get(l.item(), 0.4) for l in labels],
                             device=waveforms.device)
        do_aug = torch.rand(B, device=waveforms.device) < probs
        if not do_aug.any(): return waveforms

        aug_choices = torch.randint(0, 5, (B,), device=waveforms.device)

        gain_mask = do_aug & (aug_choices == 3)
        if gain_mask.any():
            gains = torch.empty(B, 1, device=waveforms.device).uniform_(0.6, 1.8)
            gains = torch.where(gain_mask.unsqueeze(1), gains, torch.ones_like(gains))
            waveforms = waveforms * gains

        noise_mask = do_aug & (aug_choices == 2)
        if noise_mask.any():
            snr_db      = torch.empty(B, 1, device=waveforms.device).uniform_(
                              REAL_NOISE_SNR_MIN, REAL_NOISE_SNR_MAX)
            sig_power   = waveforms.pow(2).mean(dim=1, keepdim=True)
            noise_power = sig_power / (10 ** (snr_db / 10.0))
            noise       = torch.randn_like(waveforms) * noise_power.sqrt()
            noise       = torch.where(noise_mask.unsqueeze(1), noise, torch.zeros_like(noise))
            waveforms   = waveforms + noise

        codec_mask = do_aug & (aug_choices == 4)
        if codec_mask.any():
            for i in torch.where(codec_mask)[0]:
                q = (waveforms[i] * 127).round().clamp(-127, 127) / 127.0
                waveforms[i] = q

        seq_mask = do_aug & ((aug_choices == 0) | (aug_choices == 1))
        for i in torch.where(seq_mask)[0]:
            y = waveforms[i:i+1]
            c = aug_choices[i].item()
            if c == 0:
                y = AF.pitch_shift(y, self.sr, random.uniform(-2.0, 2.0))
                y = self._ensure_len(y)
            elif c == 1:
                new_sr = int(self.sr * random.uniform(0.85, 1.15))
                y = AF.resample(y, orig_freq=self.sr, new_freq=new_sr)
                y = self._ensure_len(y)
            waveforms[i] = y.squeeze(0)

        return waveforms


class CryWaveDataset(Dataset):
    def __init__(self, entries, label_list, apply_reverb=False):
        # FIX: store entries as plain list of (path, start_sample_or_None) tuples
        # always — avoids the isinstance() branch overhead on every __getitem__
        self.entries      = [(e, None) if not isinstance(e, tuple) else e
                             for e in entries]
        self.labels       = label_list
        self.apply_reverb = apply_reverb

    def __len__(self): return len(self.entries)

    def __getitem__(self, idx):
        path, start = self.entries[idx]
        label       = self.labels[idx]
        y = load_segment(path, start_sample=start)
        if self.apply_reverb and random.random() < 0.40:
            y = add_reverb(y)
        return torch.from_numpy(peak_norm(y).astype(np.float32)), label


# FIX: val/test entries are now plain (path, None) tuples — consistent with train
val_entries  = [(p, None) for p in X_val]
test_entries = [(p, None) for p in X_test]

# Weighted sampler to handle class imbalance (burping=310 vs needs=3103)
label_counts   = Counter(train_entry_labels)
sample_weights = [1.0 / label_counts[l] for l in train_entry_labels]
sampler        = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

gpu_augmentor = GPUAudioAugmentor().to(DEVICE)

train_ds = CryWaveDataset(train_entries, train_entry_labels, apply_reverb=True)
val_ds   = CryWaveDataset(val_entries,   y_val)
test_ds  = CryWaveDataset(test_entries,  y_test)

# FIX: persistent_workers=True avoids re-spawning worker processes every epoch
# FIX: prefetch_factor=2 keeps the pipeline fed (each worker pre-loads 2 batches)
# FIX: num_workers bumped to max useful (Kaggle T4 = 4 cores, P100 = 2)
_NUM_WORKERS = min(4, os.cpu_count() or 1)
_DL_KWARGS   = dict(num_workers=_NUM_WORKERS, pin_memory=True,
                    persistent_workers=(_NUM_WORKERS > 0),
                    prefetch_factor=2 if _NUM_WORKERS > 0 else None)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, **_DL_KWARGS)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,   **_DL_KWARGS)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,   **_DL_KWARGS)

print(f"  Train entries : {len(train_ds)}  (after multi-crop)")
print(f"  Val entries   : {len(val_ds)}")
print(f"  Test entries  : {len(test_ds)}")
print(f"  num_workers   : {_NUM_WORKERS}  |  persistent_workers: {_NUM_WORKERS > 0}")


Building multi-crop index...


Building index: 100%|██████████| 3409/3409 [00:00<00:00, 603180.02it/s]


  Train entries : 3409  (after multi-crop)
  Val entries   : 731
  Test entries  : 731
  num_workers   : 4  |  persistent_workers: True


In [31]:
# ==============================================================================
# Cell 6 — ECAPA-TDNN Backbone  (unchanged — same as before)
# ==============================================================================
class SEBlock(nn.Module):
    def __init__(self, channels, bottleneck=128):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(channels, bottleneck, 1), nn.ReLU(),
            nn.Conv1d(bottleneck, channels, 1), nn.Sigmoid())
    def forward(self, x): return x * self.se(x)

class ECAPABlock(nn.Module):
    def __init__(self, channels, scale=8, dilation=1):
        super().__init__()
        self.scale, self.width = scale, channels // scale
        self.convs = nn.ModuleList([
            nn.Conv1d(self.width, self.width, 3, dilation=dilation, padding=dilation)
            for _ in range(scale - 1)])
        self.se = SEBlock(channels)
        self.bn = nn.BatchNorm1d(channels)
    def forward(self, x):
        spx = torch.split(x, self.width, 1)
        out, sp = [spx[0]], None
        for i in range(self.scale - 1):
            sp = spx[i+1] if i == 0 else sp + spx[i+1]
            sp = self.convs[i](sp)
            out.append(sp)
        return F.relu(self.bn(self.se(torch.cat(out, dim=1)) + x))

class DifferentialAttentionPool(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(channels, channels // 8), nn.ReLU(),
            nn.Linear(channels // 8, channels), nn.Sigmoid())
    def forward(self, x):
        mu   = x.mean(dim=1)
        attn = self.attention(mu)
        mu2  = mu * attn
        sig  = torch.std(x * attn.unsqueeze(1), dim=1)
        return torch.cat([mu2, sig], dim=-1)

class ECAPABackbone(nn.Module):
    """
    d_model is now 768 (wav2vec2-Base output dim) instead of 387.
    Everything else is identical to the previous version.
    """
    def __init__(self, d_model=768, channels=512, embed_dim=512):
        super().__init__()
        self.conv1  = nn.Conv1d(d_model, channels, 5, padding=2)
        self.bn1    = nn.BatchNorm1d(channels)
        self.layer1 = ECAPABlock(channels, dilation=2)
        self.layer2 = ECAPABlock(channels, dilation=3)
        self.layer3 = ECAPABlock(channels, dilation=4)
        self.conv2  = nn.Conv1d(channels * 3, 1024, 1)
        self.bn2    = nn.BatchNorm1d(1024)
        self.pool   = DifferentialAttentionPool(1024)
        self.fc1    = nn.Linear(2048, embed_dim)
        self.drop   = nn.Dropout(0.4)

    def forward(self, x):
        # x: [B, T, d_model]  (sequence-first from wav2vec2/AttentionMerge)
        x  = F.relu(self.bn1(self.conv1(x.transpose(1, 2))))
        x1 = self.layer1(x)
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        xa = F.relu(self.bn2(self.conv2(torch.cat([x1,x2,x3], dim=1)))).transpose(1,2)
        return self.drop(F.relu(self.fc1(self.pool(xa))))  # [B, 512]


In [32]:
# ==============================================================================
# Cell 7 - wav2vec2 Frontend + AttentionMerge + Full Model
# ==============================================================================
#
#  Full forward flow:
#
#  Raw waveform [B, T_samples]
#       |
#  wav2vec2-Base  (fully frozen)
#  12 transformer hidden states, each [B, T_frames, 768]
#       |
#  AttentionMerge  (trainable, ~780 params)
#  Learns per-sample, per-layer weights
#  -> [B, T_frames, 768]
#       |
#  ECAPABackbone  (d_model=768)
#  -> [B, 512]  single utterance embedding
#       |
#  ArcFace 4-cls  (single head, all samples)

# ── Fallback defaults (in case Cell 2 was not re-run after patching) ────────
SCL_TEMP     = globals().get("SCL_TEMP",     0.07)
FOCAL_GAMMA  = globals().get("FOCAL_GAMMA",  2.0)
LAMBDA_FOCAL = globals().get("LAMBDA_FOCAL", 0.3)
LAMBDA_SCL   = globals().get("LAMBDA_SCL",   0.015)
MIXUP_PROB   = globals().get("MIXUP_PROB",   0.3)
MIXUP_ALPHA  = globals().get("MIXUP_ALPHA",  0.4)

from transformers import Wav2Vec2Model

print("Loading wav2vec2-base...")
_w2v2 = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
_w2v2.eval()
for p in _w2v2.parameters():
    p.requires_grad = False
print(f"  wav2vec2-base loaded and frozen: {sum(p.numel() for p in _w2v2.parameters()):,} params")


class AttentionMerge(nn.Module):
    def __init__(self, n_layers=12, d_model=768):
        super().__init__()
        self.squeeze = nn.Linear(d_model, 1, bias=False)
        self.gate    = nn.Linear(n_layers, n_layers)

    def forward(self, hidden_states):
        stacked  = torch.stack(hidden_states, dim=2)           # [B, T, 12, 768]
        avg_t    = stacked.mean(dim=1)                         # [B, 12, 768]
        squeezed = self.squeeze(avg_t).squeeze(-1)             # [B, 12]
        attn_w   = torch.softmax(self.gate(squeezed), dim=-1)  # [B, 12]
        merged   = (stacked * attn_w.unsqueeze(1).unsqueeze(-1)).sum(dim=2)
        return merged                                          # [B, T, 768]


class AAMSoftmax(nn.Module):
    def __init__(self, in_features, n_classes, s=20.0, m=0.35):
        super().__init__()
        self.s, self.m = s, m
        self.weight = nn.Parameter(torch.FloatTensor(n_classes, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, x, label):
        cosine  = F.linear(F.normalize(x, dim=1), F.normalize(self.weight, dim=1))
        sine    = torch.sqrt((1.0 - cosine.pow(2)).clamp(min=1e-8))
        phi     = cosine * math.cos(self.m) - sine * math.sin(self.m)
        phi     = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)
        output  = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        return F.cross_entropy(output * self.s, label)

    def get_logits(self, x):
        return F.linear(F.normalize(x, dim=1), F.normalize(self.weight, dim=1)) * self.s


# ── Supervised Contrastive Loss ──────────────────────────────────────────────
# Pulls same-class embeddings together, pushes different-class apart.
# Applied on proj_head output (64-dim), not on the 512-dim ArcFace embedding,
# so the two objectives don't interfere with each other geometrically.
class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temp = temperature

    def forward(self, features, labels):
        # features: [B, D]  (will be L2-normalised inside)
        features = F.normalize(features, dim=1)         # [B, D]
        sim      = torch.matmul(features, features.T) / self.temp  # [B, B]
        B        = labels.size(0)

        # Positive mask: same class, excluding diagonal (self)
        mask_pos = (labels.unsqueeze(1) == labels.unsqueeze(0))  # [B, B]
        mask_pos.fill_diagonal_(False)

        # If a sample has no positive pair in this batch, skip it
        n_pos = mask_pos.float().sum(dim=1)             # [B]
        valid = n_pos > 0                               # [B] bool
        if not valid.any():
            return torch.tensor(0.0, device=features.device, requires_grad=True)

        # Numerically stable log-sum-exp over all other samples (negatives)
        mask_all = ~torch.eye(B, dtype=torch.bool, device=features.device)
        logits_max = sim.detach().max(dim=1, keepdim=True).values
        sim_stable = sim - logits_max                   # stability shift
        exp_sim    = torch.exp(sim_stable) * mask_all.float()
        log_denom  = torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)
        log_prob   = sim_stable - log_denom             # [B, B]

        # Mean log-prob over positives, mean over valid samples
        loss_per_sample = -(log_prob * mask_pos.float()).sum(dim=1) / n_pos.clamp(min=1)
        return loss_per_sample[valid].mean()


# ── Focal Loss ───────────────────────────────────────────────────────────────
# Down-weights easy (needs) samples so gradient budget flows to hard
# (burping, physical_pain) samples. Applied on ArcFace logits as auxiliary.
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, labels):
        # logits: raw (unscaled) [B, C] — we use get_logits() output
        ce  = F.cross_entropy(logits, labels, reduction="none")   # [B]
        pt  = torch.exp(-ce)                                       # [B]
        return ((1.0 - pt) ** self.gamma * ce).mean()


class W2VECAModel(nn.Module):
    def __init__(self, w2v2_model, embed_dim=512):
        super().__init__()
        self.w2v2      = w2v2_model
        self.merge     = AttentionMerge(n_layers=12, d_model=768)
        self.backbone  = ECAPABackbone(d_model=768, channels=512, embed_dim=embed_dim)
        # SCL projection head — small MLP mapping embedding -> contrastive space
        # Kept separate from the classification embedding so ArcFace geometry
        # is not distorted by the contrastive objective
        self.proj_head = nn.Sequential(
            nn.Linear(embed_dim, 128, bias=False),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 64, bias=False),
        )

    def _extract_features(self, x):
        with torch.no_grad():
            out = self.w2v2(x, output_hidden_states=True)
        hidden = list(out.hidden_states[1:])
        return self.merge(hidden)

    def get_embedding(self, x):
        feats = self._extract_features(x)
        return F.normalize(self.backbone(feats), dim=1)

    def forward(self, x):
        feats = self._extract_features(x)   # [B, T, 768]
        emb   = self.backbone(feats)         # [B, 512]
        return emb


# Instantiate
model    = W2VECAModel(_w2v2, embed_dim=512).to(DEVICE)
aam_head = AAMSoftmax(512, N_CLASSES, s=AAM_SCALE, m=AAM_MARGIN).to(DEVICE)

# Loss function instances
scl_criterion   = SupConLoss(temperature=SCL_TEMP).to(DEVICE)
focal_criterion = FocalLoss(gamma=FOCAL_GAMMA).to(DEVICE)

# FIX: compile ONLY the ECAPA backbone (not the full model).
# Compiling the full W2VECAModel breaks DataParallel because compile wraps
# the module in OptimizedModule, hiding .w2v2 from replicas.
# Compiling only the backbone (pure conv/linear ops) is safe and gives
# most of the speedup anyway — wav2vec2 is frozen and not the bottleneck.
if USE_COMPILE and hasattr(torch, "compile") and N_GPUS <= 1:
    _backbone = model.backbone
    model.backbone = torch.compile(_backbone, mode="reduce-overhead")
    aam_head = torch.compile(aam_head, mode="reduce-overhead")
    print("  torch.compile enabled on backbone + aam_head (single GPU)")
else:
    if N_GPUS > 1:
        print("  torch.compile skipped — DataParallel active (multi-GPU)")
    else:
        print("  torch.compile skipped")

# DataParallel
if N_GPUS > 1:
    model    = nn.DataParallel(model)
    aam_head = nn.DataParallel(aam_head)
n_frozen    = sum(p.numel() for p in _w2v2.parameters())
n_trainable = (sum(p.numel() for p in model.parameters() if p.requires_grad) +
               sum(p.numel() for p in aam_head.parameters() if p.requires_grad))
print(f"\n  wav2vec2 frozen  : {n_frozen:,} params")
print(f"  Trainable        : {n_trainable:,} params")
print(f"\n  Architecture:")
print(f"  Raw waveform")
print(f"    -> wav2vec2-Base (frozen) -> 12 hidden states [B, T, 768]")
print(f"    -> AttentionMerge         -> [B, T, 768]")
print(f"    -> ECAPA backbone         -> [B, 512]")
print(f"    -> ArcFace {N_CLASSES}-cls           loss x{LAMBDA_ARCFACE}")
print(f"    -> Focal (on same logits)         loss x{LAMBDA_FOCAL}")
print(f"    -> proj_head -> SCL (64-dim)      loss x{LAMBDA_SCL}")


# Checkpoint helpers
def _strip(sd):
    return {(k[7:] if k.startswith("module.") else k): v for k, v in sd.items()}

def save_checkpoint(epoch, val_f1):
    rm = model.module    if isinstance(model,    nn.DataParallel) else model
    ra = aam_head.module if isinstance(aam_head, nn.DataParallel) else aam_head
    torch.save({"model": rm.state_dict(), "aam": ra.state_dict(),
                "epoch": epoch, "best_f1": val_f1}, MODEL_PATH)

def load_checkpoint(path):
    if not os.path.exists(path): print("No checkpoint found."); return
    ck = torch.load(path, map_location=DEVICE)
    (model.module    if isinstance(model,    nn.DataParallel) else model   ).load_state_dict(_strip(ck["model"]))
    (aam_head.module if isinstance(aam_head, nn.DataParallel) else aam_head).load_state_dict(_strip(ck["aam"]))
    print(f"  Loaded checkpoint (epoch {ck.get('epoch','?')}, best F1 {ck.get('best_f1',0):.4f})")


# Optimizer + scheduler with warmup
optimizer = torch.optim.AdamW(
    [p for p in list(model.parameters()) + list(aam_head.parameters()) if p.requires_grad],
    lr=LR, weight_decay=WEIGHT_DECAY)

def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
print("\n  Optimizer and scheduler ready.")


Loading wav2vec2-base...


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  wav2vec2-base loaded and frozen: 94,371,712 params
  torch.compile skipped — DataParallel active (multi-GPU)

  wav2vec2 frozen  : 94,371,712 params
  Trainable        : 5,590,492 params

  Architecture:
  Raw waveform
    -> wav2vec2-Base (frozen) -> 12 hidden states [B, T, 768]
    -> AttentionMerge         -> [B, T, 768]
    -> ECAPA backbone         -> [B, 512]
    -> ArcFace 4-cls           loss x1.0
    -> Focal (on same logits)         loss x0.3
    -> proj_head -> SCL (64-dim)      loss x0.015

  Optimizer and scheduler ready.


In [34]:
# ==============================================================================
# Cell 8 - Training Loop
# ==============================================================================
# Loss = ArcFace  (margin-based, main classification signal)
#      + Focal    (auxiliary on same logits, re-weights hard/minority samples)
#      + SCL      (contrastive on proj_head output, tightens intra-class clusters)
# MixUp is applied at batch level before the forward pass.
# MixUp samples are excluded from SCL (contrastive loss needs clean labels).
# ==============================================================================

train_losses, val_f1_scores = [], []
best_val_f1, patience_ctr   = 0.0, 0

# GradScaler for AMP
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))


def apply_mixup(x, labels, alpha=MIXUP_ALPHA):
    lam     = float(np.random.beta(alpha, alpha))
    idx     = torch.randperm(x.size(0), device=x.device)
    x_mix   = lam * x + (1.0 - lam) * x[idx]
    return x_mix, labels, labels[idx], lam


def get_base(mod, wrapped):
    return mod.module if isinstance(mod, nn.DataParallel) else mod


print("=" * 90)
print("  wav2vec2-Base -> AttentionMerge -> ECAPA  |  ArcFace + Focal + SCL + MixUp".center(90))
print("=" * 90)
print(f"  ArcFace x{LAMBDA_ARCFACE}  |  Focal x{LAMBDA_FOCAL} (gamma={FOCAL_GAMMA})  |  "
      f"SCL x{LAMBDA_SCL} (T={SCL_TEMP})  |  MixUp p={MIXUP_PROB}")
print(f"  Warmup: {WARMUP_EPOCHS} epochs  |  Batch: {BATCH_SIZE}  |  LR: {LR}")
print("=" * 90)

for epoch in range(1, EPOCHS + 1):
    model.train(); aam_head.train()
    epoch_loss = epoch_arc = epoch_foc = epoch_scl = epoch_acc = n_batches = 0

    for waveforms, labels in tqdm(train_loader, desc=f"Epoch {epoch:03d}", leave=False, ncols=90):
        if torch.isnan(waveforms).any(): continue
        waveforms, labels = waveforms.to(DEVICE), labels.to(DEVICE)

        # ── GPU augmentation ──────────────────────────────────────────────────
        waveforms = gpu_augmentor(waveforms, labels, AUG_PROB_BY_IDX)
        x = (waveforms - waveforms.mean(1, keepdim=True)) / (waveforms.std(1, keepdim=True) + 1e-8)

        # ── MixUp (applied to ~30% of batches) ───────────────────────────────
        do_mixup   = random.random() < MIXUP_PROB
        labels_b   = None
        lam        = 1.0
        if do_mixup:
            x, labels, labels_b, lam = apply_mixup(x, labels)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            embeddings = model(x)                               # [B, 512]

            # ── 1. ArcFace loss ───────────────────────────────────────────────
            if do_mixup:
                # MixUp: interpolate loss between the two label sets
                arc_loss = (lam        * aam_head(embeddings, labels) +
                            (1.0-lam)  * aam_head(embeddings, labels_b))
            else:
                arc_loss = aam_head(embeddings, labels)
            if arc_loss.dim() > 0: arc_loss = arc_loss.mean()

            # ── 2. Focal loss (on ArcFace logits, no gradient through margin) ─
            base_aam = get_base(aam_head, None)
            logits   = base_aam.get_logits(embeddings)          # [B, C]
            if do_mixup:
                foc_loss = (lam       * focal_criterion(logits, labels) +
                            (1.0-lam) * focal_criterion(logits, labels_b))
            else:
                foc_loss = focal_criterion(logits, labels)

            # ── 3. SCL loss (only on clean — non-MixUp — batches) ─────────────
            # MixUp creates mixed labels so contrastive positives are undefined.
            # Skipping MixUp batches for SCL is the standard approach.
            if not do_mixup:
                base_model = get_base(model, None)
                proj       = base_model.proj_head(embeddings)   # [B, 64]
                scl_loss   = scl_criterion(proj, labels)
            else:
                scl_loss   = torch.tensor(0.0, device=DEVICE)

            loss = (LAMBDA_ARCFACE * arc_loss +
                    LAMBDA_FOCAL   * foc_loss +
                    LAMBDA_SCL     * scl_loss)

        if not torch.isfinite(loss): continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(
            [p for p in list(model.parameters()) + list(aam_head.parameters())
             if p.requires_grad], 1.0)
        scaler.step(optimizer)
        scaler.update()

        with torch.no_grad():
            base_aam_ng = get_base(aam_head, None)
            preds = base_aam_ng.get_logits(F.normalize(embeddings.float(), dim=1)).argmax(1)
            epoch_acc += (preds == labels).float().mean().item()

        epoch_loss += loss.item()
        epoch_arc  += arc_loss.item()
        epoch_foc  += foc_loss.item()
        epoch_scl  += scl_loss.item()
        n_batches  += 1

    scheduler.step()
    nb_    = max(n_batches, 1)
    avg_loss = epoch_loss / nb_
    avg_acc  = epoch_acc  / nb_
    train_losses.append(avg_loss)

    # ── Validation ────────────────────────────────────────────────────────────
    model.eval(); aam_head.eval()
    val_preds, val_trues = [], []
    base_model_v = get_base(model,    None)
    base_aam_v   = get_base(aam_head, None)

    with torch.no_grad():
        for waveforms, labels in val_loader:
            if torch.isnan(waveforms).any(): continue
            x = waveforms.to(DEVICE)
            x = (x - x.mean(1, keepdim=True)) / (x.std(1, keepdim=True) + 1e-8)
            with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                emb   = base_model_v(x)
                preds = base_aam_v.get_logits(emb).argmax(1)
            val_preds.extend(preds.cpu().tolist())
            val_trues.extend(labels.tolist())

    val_f1 = f1_score(val_trues, val_preds, average="macro", zero_division=0)
    val_f1_scores.append(val_f1)
    lr_now = scheduler.get_last_lr()[0]

    if val_f1 > best_val_f1:
        best_val_f1, patience_ctr = val_f1, 0
        save_checkpoint(epoch, val_f1)
        status = "BEST MODEL SAVED"
    else:
        patience_ctr += 1
        status = f"PATIENCE {patience_ctr}/{PATIENCE}"

    print(f"Epoch {epoch:03d}/{EPOCHS} | "
          f"Loss: {avg_loss:.4f} "
          f"[arc={epoch_arc/nb_:.3f} foc={epoch_foc/nb_:.3f} scl={epoch_scl/nb_:.3f}] | "
          f"Acc: {avg_acc:6.2%} | F1: {val_f1:.4f} | LR: {lr_now:.1e} | {status}")

    if patience_ctr >= PATIENCE:
        print("=" * 90)
        print(f"Early stopping at epoch {epoch}. Best Val F1: {best_val_f1:.4f}")
        print("=" * 90)
        break


         wav2vec2-Base -> AttentionMerge -> ECAPA  |  ArcFace + Focal + SCL + MixUp       
  ArcFace x1.0  |  Focal x0.3 (gamma=2.0)  |  SCL x0.015 (T=0.07)  |  MixUp p=0.3
  Warmup: 5 epochs  |  Batch: 64  |  LR: 0.0003


Epoch 001/100 | Loss: 7.5780 [arc=7.290 foc=0.766 scl=3.855] | Acc: 42.12% | F1: 0.5342 | LR: 1.2e-04 | BEST MODEL SAVED


Epoch 002/100 | Loss: 6.1486 [arc=5.831 foc=0.894 scl=3.320] | Acc: 58.29% | F1: 0.6779 | LR: 1.8e-04 | BEST MODEL SAVED


Epoch 003/100 | Loss: 5.4303 [arc=5.126 foc=0.852 scl=3.238] | Acc: 64.01% | F1: 0.6095 | LR: 2.4e-04 | PATIENCE 1/20


Epoch 004/100 | Loss: 4.9022 [arc=4.609 foc=0.831 scl=2.914] | Acc: 66.95% | F1: 0.6299 | LR: 3.0e-04 | PATIENCE 2/20


...

Epoch 065/100 | Loss: 1.5297 [arc=1.315 foc=0.615 scl=2.000] | Acc: 86.67% | F1: 0.8324 | LR: 9.0e-05 | PATIENCE 16/20


Epoch 066/100 | Loss: 1.1656 [arc=0.992 foc=0.471 scl=2.142] | Acc: 85.46% | F1: 0.8277 | LR: 8.5e-05 | PATIENCE 17/20


Epoch 067/100 | Loss: 1.3385 [arc=1.142 foc=0.549 scl=2.102] | Acc: 87.34% | F1: 0.8350 | LR: 8.1e-05 | PATIENCE 18/20


Epoch 068/100 | Loss: 1.1744 [arc=0.994 foc=0.500 scl=2.026] | Acc: 86.28% | F1: 0.8323 | LR: 7.6e-05 | PATIENCE 19/20


Epoch 069/100 | Loss: 1.1815 [arc=1.002 foc=0.486 scl=2.252] | Acc: 87.22% | F1: 0.8311 | LR: 7.2e-05 | PATIENCE 20/20
Early stopping at epoch 69. Best Val F1: 0.8419


In [36]:
path = "/kaggle/working/checkpoints"

print("Exists:", os.path.exists(path))
if os.path.exists(path):
    print(os.listdir(path))

Exists: True
['best_w2v_ecapa.pt']


In [38]:
# ==============================================================================
# Cell 9 - Test Set Evaluation
# ==============================================================================
load_checkpoint(MODEL_PATH)
model.eval(); aam_head.eval()

all_preds, all_trues = [], []
base_model = model.module    if isinstance(model,    nn.DataParallel) else model
base_aam   = aam_head.module if isinstance(aam_head, nn.DataParallel) else aam_head

with torch.no_grad():
    for waveforms, labels in tqdm(test_loader, desc="Test", ncols=70):
        if torch.isnan(waveforms).any(): continue
        x = waveforms.to(DEVICE)
        x = (x - x.mean(1, keepdim=True)) / (x.std(1, keepdim=True) + 1e-8)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == 'cuda')):
            emb   = base_model(x)
            preds = base_aam.get_logits(emb).argmax(1)
        all_preds.extend(preds.cpu().tolist())
        all_trues.extend(labels.cpu().tolist())

acc = sum(p == t for p, t in zip(all_preds, all_trues)) / len(all_trues)
f1  = f1_score(all_trues, all_preds, average="macro", zero_division=0)
print(f"\nTest Acc : {acc:.4f}")
print(f"Test F1  : {f1:.4f}")
print(classification_report(all_trues, all_preds, target_names=CLASSES, zero_division=0))


  Loaded checkpoint (epoch 49, best F1 0.8419)


Test: 100%|███████████████████████████| 12/12 [00:08<00:00,  1.41it/s]


Test Acc : 0.8714
Test F1  : 0.8099
               precision    recall  f1-score   support

       scared       0.99      0.99      0.99       103
        needs       0.92      0.89      0.91       466
physical_pain       0.72      0.77      0.74       116
      burping       0.57      0.63      0.60        46

     accuracy                           0.87       731
    macro avg       0.80      0.82      0.81       731
 weighted avg       0.88      0.87      0.87       731



In [40]:
# ==============================================================================
# Cell 10 - Real-World Inference
# ==============================================================================
import glob
from collections import Counter as _Counter

def rms_normalize(wav_np, target_db=-23.0):
    rms  = np.sqrt(np.mean(wav_np ** 2) + 1e-12)
    gain = 10 ** ((target_db - 20 * np.log10(rms)) / 20)
    return wav_np * gain

def energy_vad(wav, sr, frame_ms=30, hop_ms=10,
               energy_threshold_db=-38.0, min_speech_ms=300, context_ms=100):
    frame_len  = int(sr * frame_ms  / 1000)
    hop_len    = int(sr * hop_ms    / 1000)
    min_frames = max(1, int(min_speech_ms / hop_ms))
    ctx_frames = max(1, int(context_ms   / hop_ms))
    n_frames   = (len(wav) - frame_len) // hop_len + 1
    if n_frames <= 0: return [(0, len(wav))]
    energy    = np.array([np.sqrt(np.mean(wav[i*hop_len:i*hop_len+frame_len]**2)+1e-12)
                          for i in range(n_frames)])
    is_speech = 20 * np.log10(energy) > energy_threshold_db
    for i in range(ctx_frames, len(is_speech)-ctx_frames):
        if is_speech[max(0,i-ctx_frames):i+ctx_frames].any(): is_speech[i] = True
    segments, in_seg, seg_start = [], False, 0
    for i, v in enumerate(is_speech):
        if v and not in_seg:  in_seg = True;  seg_start = i
        elif not v and in_seg:
            in_seg = False
            if i - seg_start >= min_frames:
                segments.append((seg_start*hop_len, min(i*hop_len+frame_len, len(wav))))
    if in_seg and len(is_speech)-seg_start >= min_frames:
        segments.append((seg_start*hop_len, len(wav)))
    return segments if segments else [(0, len(wav))]

def preprocess_audio_file(path, target_sr=SR):
    try:
        wav, sr = torchaudio.load(path)
    except Exception:
        import librosa as lb
        y, sr = lb.load(path, sr=None, mono=True)
        wav = torch.from_numpy(y).unsqueeze(0)
    if wav.shape[0] > 1: wav = wav.mean(dim=0, keepdim=True)
    if sr != target_sr:  wav = AF.resample(wav, sr, target_sr)
    wav_np = wav.squeeze().numpy().astype(np.float64)
    wav_np = rms_normalize(wav_np)
    peak   = np.abs(wav_np).max()
    if peak > 1e-8: wav_np /= peak
    return wav_np.astype(np.float32)


@torch.no_grad()
def classify_segment(segment):
    chunk = segment[:TARGET_SAMPLES] if len(segment) >= TARGET_SAMPLES else \
            np.pad(segment, (0, TARGET_SAMPLES - len(segment)))
    x   = torch.from_numpy(chunk).unsqueeze(0).to(DEVICE)
    x   = (x - x.mean()) / (x.std() + 1e-8)
    bm  = model.module    if isinstance(model,    nn.DataParallel) else model
    ba  = aam_head.module if isinstance(aam_head, nn.DataParallel) else aam_head
    emb   = bm(x)
    logits = ba.get_logits(emb).squeeze(0)
    probs  = torch.softmax(logits, dim=0)
    conf, idx = probs.max(0)
    return CLASSES[idx.item()], conf.item()


def infer_file_summary(audio_path):
    wav  = preprocess_audio_file(audio_path)
    segs = energy_vad(wav, SR)
    results = []
    for seg_start, seg_end in segs:
        seg_wav = wav[seg_start:seg_end]
        seg_t0  = seg_start / SR
        votes   = []
        offset  = 0
        while offset < len(seg_wav):
            chunk = seg_wav[offset:offset+TARGET_SAMPLES]
            if len(chunk) >= int(0.4 * TARGET_SAMPLES):
                label, conf = classify_segment(chunk)
                votes.append((label, conf))
            offset += SR
        if not votes: continue
        winner  = _Counter(v[0] for v in votes).most_common(1)[0][0]
        w_votes = [v for v in votes if v[0] == winner]
        results.append({
            "start_sec":  seg_t0,
            "end_sec":    seg_t0 + len(seg_wav)/SR,
            "label":      winner,
            "confidence": sum(v[1] for v in w_votes)/len(w_votes),
            "n_windows":  len(votes),
        })
    return results


SUPPORTED_EXTS = (".wav", ".mp3", ".ogg", ".m4a", ".flac", ".opus", ".aac")
INFERENCE_DIR  = "/kaggle/input/datasets/yousefradwan25/whatssapdata/Test_Data"   # update as needed

load_checkpoint(MODEL_PATH)
model.eval(); aam_head.eval()

if os.path.exists(INFERENCE_DIR):
    files = sorted([f for f in glob.glob(os.path.join(INFERENCE_DIR, "*"))
                    if f.lower().endswith(SUPPORTED_EXTS)])
    print(f"Found {len(files)} file(s) in {INFERENCE_DIR}\n")
    for fpath in files:
        print(f"{chr(9472)*65}")
        print(f"  File : {os.path.basename(fpath)}")
        try:
            results = infer_file_summary(fpath)
            if not results:
                print("  (no speech detected)")
            else:
                for r in results:
                    print(f"  [{r['start_sec']:5.1f}s-{r['end_sec']:5.1f}s]  "
                          f"{r['label'].upper():15s}  "
                          f"conf={r['confidence']:.2f}  "
                          f"({r['n_windows']} windows)")
        except Exception as e:
            print(f"  ERROR: {e}")
    print(f"{chr(9472)*65}")
else:
    print(f"INFERENCE_DIR not found: {INFERENCE_DIR}")
    print("  Update INFERENCE_DIR at the top of this cell.")


  Loaded checkpoint (epoch 49, best F1 0.8419)
Found 7 file(s) in /kaggle/input/datasets/yousefradwan25/whatssapdata/Test_Data

─────────────────────────────────────────────────────────────────
  File : AUD-20260501-WA0000.opus
  [  0.2s- 24.9s]  NEEDS            conf=0.99  (23 windows)
─────────────────────────────────────────────────────────────────
  File : AUD-20260501-WA0001.opus
  [  0.1s- 12.8s]  NEEDS            conf=0.95  (11 windows)
─────────────────────────────────────────────────────────────────
  File : AUD-20260501-WA0002.opus
  [  0.4s-  8.1s]  BURPING          conf=0.98  (6 windows)
─────────────────────────────────────────────────────────────────
  File : AUD-20260501-WA0003.opus
  [  0.3s-  8.2s]  BURPING          conf=0.84  (6 windows)
─────────────────────────────────────────────────────────────────
  File : AUD-20260501-WA0005.opus
  [  0.7s- 64.2s]  PHYSICAL_PAIN    conf=0.92  (62 windows)
─────────────────────────────────────────────────────────────────
  File :

In [41]:
# ==============================================================================
# Cell 12 — Download Checkpoint from Kaggle
# ==============================================================================
import os
from IPython.display import FileLink, display

CKPT_PATH = MODEL_PATH   # defined in Cell 2: /kaggle/working/checkpoints/best_w2v_ecapa.pt

if not os.path.exists(CKPT_PATH):
    print(f"No checkpoint found at {CKPT_PATH}")
    print("Make sure Cell 8 (training) completed and saved a best model.")
else:
    size_mb = os.path.getsize(CKPT_PATH) / (1024 ** 2)
    print(f"Checkpoint found: {CKPT_PATH}")
    print(f"Size            : {size_mb:.1f} MB")
    print(f"Epoch           : {torch.load(CKPT_PATH, map_location='cpu').get('epoch', '?')}")
    print(f"Best Val F1     : {torch.load(CKPT_PATH, map_location='cpu').get('best_f1', 0):.4f}")
    print()

    # ── Option 1: click the link below to download directly ──────────────────
    display(FileLink(CKPT_PATH, result_html_prefix="Download checkpoint: "))

    # ── Option 2: copy to /kaggle/working root for easier access ─────────────
    import shutil
    dest = "/kaggle/working/best_w2v_ecapa.pt"
    shutil.copy2(CKPT_PATH, dest)
    print(f"\nAlso copied to: {dest}")
    display(FileLink(dest, result_html_prefix="Or download from root: "))

Checkpoint found: /kaggle/working/checkpoints/best_w2v_ecapa.pt
Size            : 381.5 MB
Epoch           : 49
Best Val F1     : 0.8419



/kaggle/working/checkpoints/best_w2v_ecapa.pt


Also copied to: /kaggle/working/best_w2v_ecapa.pt


/kaggle/working/best_w2v_ecapa.pt

In [42]:
from IPython.display import FileLink
FileLink(r'/kaggle/working/checkpoints/best_w2v_ecapa.pt')

/kaggle/working/checkpoints/best_w2v_ecapa.pt

In [43]:
import os
# Double check the file exists in the root working directory
if os.path.exists('best_w2v_ecapa.pt'):
    print("File is ready for download!")
else:
    # If it's only inside the checkpoints folder, copy it out
    import shutil
    shutil.copy('/kaggle/working/checkpoints/best_w2v_ecapa.pt', '/kaggle/working/best_w2v_ecapa.pt')
    print("Copied to root successfully.")

File is ready for download!
